# Heart Disease Prediction Using PSO-Optimized Random Forest
**Pipeline:** Data → Preprocessing → Baseline RF → PSO-tuned RF → Comparison & Visualisation

Dataset: UCI Heart Disease (Cleveland subset, 303 patients, 13 features).

## 1. Setup & Dataset Loading
The Cleveland dataset is embedded in the notebook, so just run the cell — no upload required.

In [ ]:
import os, zipfile, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
warnings.filterwarnings("ignore")

SEED = 42                      # fixed seed -> reproducible results
random.seed(SEED); np.random.seed(SEED)
sns.set_style("whitegrid")

In [ ]:
#@title Load dataset (Cleveland data stored inside the notebook) {display-mode: "form"}
import base64, zlib
os.makedirs("heart_disease", exist_ok=True)
with open("heart_disease/processed.cleveland.data", "wb") as f: f.write(zlib.decompress(base64.b64decode("eNqVXNtyIysMfM+3bG2hC5d52v//q+MMY1BLwsl5yG7iMoYBqdVqCTf5W/7Q+0fr61+W92v8/X8tr3/L/Zf8kef39v3/V+vPG/X7/3a/abT19tdrZayPr/cr8vywG8z3EL7s4PuvuY52v/L9c4/6kvfge6lyD1kL/f6h0Z/f5DWzrN+/l61kZ5mDi9rXOr8/5q/+IRhcm92cuWxpduY+1l/DDW7vz72fWe/BbdiZW1kD2tyo96Z91W4H3zNLVTtzex9ceQ12M4vd7WfD4Jm1m2fm5939PqoKg+eyCxpJXedBa7fnUdUeBtPFdtm6N2zOvC2sgjXNmS9YdpW1bFmD3VFZI4HjUzYb9v693c+sGs+5CZwz/GWe+TUz25lva6LrMm+jtneg+sFg23W+fcDMumb156zDLpvQq/Cj3lOYo9J4zoKD9/NyNnPB3e7VDjYfFQZfdtlzcEOvovVXsG21GDafmcDHdb9jeCOx2zF3mwfYdmP7jjjYGcnQ1J/9zLQHzw27AY4RhjZu8Zqm32DQSvTnAs4ivNamgJ76VYtdyFw2wTmbXWmnDduDRQF6OyeO8RxVi7vNcM60d6B53AYYmoMXaqGRkPGqZzBsGE1r7nbDyHo8G8d47fZll505RiVzzmRwu7ijohtSpQAYbOtSOGfeti1rZpGaHpXf7ZeFXWAkM7hjxCADQwwwpBrRUxAAN6L5KLmA1SAJn6KkNxI8Z55vh6Piso6quqPCcDONpINj0An0XxtWYcPuM6zwzMurvgHQRsmXkVDAbVax57xYCxkLe4yk2uD+OEYN5lnMYFn+rAVIXFnhBnebHKGZ59wp4JUU8GATq9TZ9rIw88xmf79f68dYtYxvY5hIgdVcZ9sG9LznZDd4b1+Fo3oBIBzVBAOMVSzHcFNLAkMC5jkMhlnz1H3O25+VOlCp7ugj7ZkluCRdOJhNfP7Eestt6ddImYE1J2OeJ5cs61GKAwOHJLqP6iopifMzex4mGKvu18YZSbIoiXMZZqBr5owN3ZF5c5K5iZu1IBjQNhIxEUMCWS8OhuZgpRirVlDlI+i7Zc/4fIN+B/OslnsKWhhZJJmDCW27JhtmBjt/LuBV6tGzrECnDRyDI+hDlJSch+lyPyZkQ+SOSnDDNnGt97LVYlgvRwtzRlKiQZIc8yo0z5l8Dzhn3SRuAIkzg7eF0YCIUS3WOO6pEbeFU/S0uN0R9HVtjnBM1ObHK+QYzzn7mQmMpJsUGJeN6Hk/H2OM0A/JNwoOdaXAlLgkw27LzNydWtEsAK70jtxJkQPPmZ/oOLhzW2A3KYkDz435O0voB/D0+zVje0PkpfN+lWS/YNlKzjoN/lGIc1JGMG3L1emQxE7YVjDtUdxJnbDgIYDwzLQTuQGmLV8KR3WjALtlW97hZpaAf9xQY5FE6ZiD5YpEnyFkjw9pFZChMZ9Zg4pHLqF7GDOHJFa4HhK6kLhrEudORxWIPqp4N2xXsO3azsgrkIcmmE81GfwvHtSAvAjT9mReijqHk6muRNsx+S/CyCbb3iG9iaB6ON3T2Vdd6hkvqdUBmE3nEHebkyoEGdx65inqKgw27NkLBk1CfOXKqdya7DbHyF5rSGHJwV9P8l/CtJ1TQuKCs8vIEP6MqTYXLtCnZOIIHdL2cYK/+cxTGIING2e3QASjGSIB/kY9DhZU0gbADoP0mBwV6tPjtm0kntcxwejgVfOoVtzcYSAfrICdGQmDU2dwjAo4WeayUdtpZ6H3iuI2xuPGB/NkJ/RO28YKSC0ODLZtN0oISQkZGRketWcWdIwRBUC1Qq8Tw4Dm6+TLgNqL45FT0mhr8oZUoPhrMrLtkg9r5cQ8AXovuwNYSlAE/dsIlMDCWhLaHzDgCPpQDbi626+92W6SuqonKCrlq0aZQ7e2TYGoJ+5cQ5yjATOPc6ErqRiJFHgUNpSXMNrUqDxiuYm3xQy/YSPJ2lvKZux6npklwLaida51aBDSUBcqBZC3LAbnM9gH/2rUPAmwYB1cNdq22zDalGRAEDFFH3V+gdz4ie1YSTiH54VuM0jNGgYScGsJgnFOP0TYKaS1Q44gaYmNsZJwpmFIhqY6g7Dd5Oes3RRAXGrTExrWUSP5HnwXI5nhmDGpNI9cvjipc0HpmQt/Zp7OwBoFXkBnUX3TZQZlpzggSU27IIRVFFvJWece3KPK4WgEV1PoF9RLOamwoZJ50ldk4p8TZ1DylFNaRA4LnkgFjMSIgf6ZK4a0adrA/5oej6peFgv69siMSLFfNm7YDSSl2rnGmcK1qB06LiSUVFJ7osiXuWElIC85FOpJKnizMFYwz8rnWgD2oUwdDU6APxRPKBYSsJWkXz+zbTacF47qE4RBGfZ2fKGRltsj50UuVKeoPkC1HAcK95TnFnjeM3MDHzdRL+RzHaj697DaTkVvf84YJJ+WoSvksG91Bss2LYqtrJb/Xc2ovFbZFodCk85gUlUu586yDSxBXldgEwMklPuFGqcCv+jkjtlohyWZWVIDIydaspOFJs3H7KT6MLVDlQLjupPXXWz6KS/qPXlmtBGfF713W50g//Rz6P9tT9BN4VBsZXY7nXTPyCYVFcCz1+SoBPt2doLBpYcCG/2iY2jyP8bw7FHonyUz2PbhCFy9EkJiUoR9UC2WIGo7LNrzTkqUMxlJZuNowdamnZZPO5RdLrK7jjSG/gIv8qZJ+zKRm4qwXinLL6FykrTwietI6wnuuvhqCj7E4fisUPGuBKjvbOAl8r6XPcZBzCIno83wKtCbYAgvQTJXXM4+sz7scio7DJJrqdBYsxYUZk1/1fVzwZsRl0ITnpxkNIX9QgJXYhtJg9RZ7zQyajP0LBtxVyE26yQzmI959Y9P7G/E+oXutJ6BLhcvWg4owbLr+PE0qkms4V31IxPi2PFjUiqFNP1S88xu2XCmDQN7gY6fYmadnZoo4F28Ovi2KlQTGvXE11h28X2x7SAdsivyz9zuar/NQTNOAeyPt9OyEy2xRWpivkIniIGw6ZL/3k/MsfhxQTJSmnFI17Q9ks5BSosuZHwqK8AWZFHYXFCMT8luwVtu0bbGQVDiL+aJCWm6L/i00BNBefJbQuvZ9maPYPQxl5vL7rGh4pDXgDdTYpz1+pzLrcEjKevVDylo0oPMQAS6uJrgZo5NY+0XpQaRpKDYk+Y/18xbjgjWk+LHDUmCqK09wZEH8pMmknoFZc0KxEYIQ56+tbR9zs3Ztgk2Nel+hnS31l8KYTUxz2p7nJDNSE/8uf5SgpMBzUJJSXFsNxyuEQ7TbkGSz9COxrFbEqLzjJEM5Br01ENtTpPOwc9pt6lfkJGUJCQ2Fj0LplRImrb691PrtLaIYSoHDCsZ8fStWRKCjSeekhS55sz9t60JioU9jldeOp9hKNadGZu2Wz+b52X+nBHO5ezX5+aCjQUCadFd76nOtk4ViGfRV0jlfK/Q45DgAk9lDszR3J3pp34KY9loX0ZSo6w9avdWcRQLgnHSoQ2OkusIZTjj3NIMNkXMZSdZEZll7/7QipW5BhzMNxHTSde2MbLbJ6RuOtnsYHZV51mN1CtAvmX5W1HCRIQx+52vkWEkyW5jW5frLmjyy6OahQAsotKug7JjYS4VTG7LFCuEHVr/rHx3hXs6JenMoi984+yoQN2s8VFr0KSKcNXj4ATB3JUqdwVDfoa/bRBSfpsi6JU0lP22xtWglXduGF7+MPJNg3NmJ00z3iEob6X1MHFysVMskHQ6CHDsOFgpcdFGdCTXT+HgD2+gbTyF44y9BaYxgfh4cY7y7lBbdBmhBFQ+c22LnZTWUG1O1ZJux9vDGGsIkP67HhKNjBctylSOUFLyfXDOp/xNhvbxig/VyEcGHVU05ThzuY5XMw/VC12prhNadZ+6OJGjt5DZ0IXxoplU8CB4blnVdbWq/GwkbC4jYPseuWUb7ERBfVJHbGu4zGDMnbEufgdYoSu9dOJVIflqJd73c5cF97Kvj72lnFwia9fP8Gd6X/D2BtekP2pWIPD62vPMNS1Z2zpo24zEeRXeJ6hZf9S/76HYyLa9eV8h27yB/Y3lEvVhzn0qkfJje6jr3K3ll3v93A8Hmm5SxcsrcC1WBHmEwe9vICBs6mphZkG1gMuhL593z+HmnYR1AK7nOtMVw2vRA6VAjZfcjeVZ4eee3t2IpfKGIHQffh+HCz7FeTMkRe86UznW2fkDCFVNbo3oOXMegcxs7CxOwGunmXeMdNdgoc7AOS+Q0y2wT5lgDV2tXKp9QmiKOzSg7MKj++6DD/20zqtG7BsMSpjpLQVecAsru7eAgJHoc0dHdqUJjqrvy6gUOjztvaSGM2+QdRZmWL4aadp/t4Zg1wwf+W4/fjEHhhqIFpJzVpP8Ygpq4C80vtSQqwqGV3Pt/XiBwvQouStNavZaINS0+MUc+4tPsJtWHWllp1OwJg7ZjmVQTIq4OL73WRNCQWmCYbfHd3k28TYv/y0oM9bgVQSWRMo3BWv3TREIWcKHphna3RRb4p3SHwUKJu4rF9g3B3i8e5Pg/GY53omREssXACOY2MgInQWuiyHUUN+dBf8B3FE8NA==")))
print(os.listdir("heart_disease"))

In [ ]:
import glob
# Find the file even if the zip extracted into a sub-folder
path = glob.glob("heart_disease/**/processed.cleveland.data", recursive=True)[0]
cols = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","target"]
df = pd.read_csv(path, header=None, names=cols, na_values="?")
print("Shape:", df.shape)
df.head()

## 2. Problem Definition & Preprocessing
**Problem:** binary classification — does the patient have heart disease?

Original target is 0 (no disease) to 4 (severity). It is converted to **0 = healthy, 1 = disease** (values 1–4).

Preprocessing: missing-value handling (median imputation), stratified 80/20 train/test split, feature scaling.

In [ ]:
# Binarise target
df["target"] = (df["target"] > 0).astype(int)

print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0], "\n")
print("Class distribution:\n", df["target"].value_counts(), "\n")
df.describe().T

In [ ]:
# ---- EDA plots ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(x="target", data=df, ax=ax[0], palette="Set2")
ax[0].set_title("Class Distribution (0 = No disease, 1 = Disease)")
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1], annot_kws={"size": 7})
ax[1].set_title("Correlation Matrix")
plt.tight_layout(); plt.show()

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

# Stratified split BEFORE any fitting -> no data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

# Imputer + scaler are fitted on training data only (inside a Pipeline)
preprocess = [("imputer", SimpleImputer(strategy="median")),
              ("scaler",  StandardScaler())]
print("Train:", X_train.shape, " Test:", X_test.shape)

## 3. Baseline Random Forest (no optimisation)
Random Forest is suitable here: small tabular dataset, mixed feature types, non-linear relationships, robust to noise, and has several hyperparameters worth tuning. Baseline uses default scikit-learn settings.

In [ ]:
def evaluate(model, Xte, yte):
    pred  = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    return {"Accuracy":  accuracy_score(yte, pred),
            "Precision": precision_score(yte, pred),
            "Recall":    recall_score(yte, pred),
            "F1-score":  f1_score(yte, pred),
            "ROC-AUC":   roc_auc_score(yte, proba)}

baseline_rf = Pipeline(preprocess + [("rf", RandomForestClassifier(random_state=SEED))])
t0 = time.time()
baseline_rf.fit(X_train, y_train)
baseline_time = time.time() - t0

base_metrics = evaluate(baseline_rf, X_test, y_test)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
base_cv = cross_val_score(baseline_rf, X_train, y_train, cv=cv, scoring="f1").mean()

print("BASELINE – test set metrics")
for k, v in base_metrics.items(): print(f"  {k:10s}: {v:.4f}")
print(f"  5-fold CV F1 (train): {base_cv:.4f}")

## 4. PSO — Particle Swarm Optimization (implemented from scratch)
Each particle is a candidate set of Random Forest hyperparameters:

| Dimension | Hyperparameter | Range |
|---|---|---|
| 0 | `n_estimators` | 50 – 300 |
| 1 | `max_depth` | 3 – 20 |
| 2 | `min_samples_split` | 2 – 20 |
| 3 | `min_samples_leaf` | 1 – 10 |
| 4 | `max_features` | 0.2 – 1.0 |

**Fitness** = mean 5-fold cross-validated F1 on the *training set only* (test set is never touched during optimisation).

**Operators:**
- Velocity: `v = w·v + c1·r1·(pbest − x) + c2·r2·(gbest − x)`
- Position: `x = x + v` (clipped to bounds)
- Linearly decreasing inertia weight `w`, personal & global best memory.

In [ ]:
# ---------------- Search space ----------------
param_names = ["n_estimators", "max_depth", "min_samples_split", "min_samples_leaf", "max_features"]
lower = np.array([50,   3,  2,  1, 0.2])
upper = np.array([300, 20, 20, 10, 1.0])
DIM = len(lower)

def decode(position):
    """Convert continuous particle position -> valid RF hyperparameters."""
    return {"n_estimators":      int(round(position[0])),
            "max_depth":         int(round(position[1])),
            "min_samples_split": int(round(position[2])),
            "min_samples_leaf":  int(round(position[3])),
            "max_features":      float(position[4])}

# ---------------- Fitness function ----------------
cv_fit = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def fitness(position):
    params = decode(position)
    model = Pipeline(preprocess + [("rf", RandomForestClassifier(random_state=SEED, n_jobs=-1, **params))])
    return cross_val_score(model, X_train, y_train, cv=cv_fit, scoring="f1").mean()

# ---------------- PSO ----------------
def pso(n_particles=12, n_iter=20, w_max=0.9, w_min=0.4, c1=1.6, c2=1.6, seed=SEED):
    rng = np.random.RandomState(seed)
    v_max = 0.2 * (upper - lower)                      # velocity clamp

    # Initialise swarm
    pos = rng.uniform(lower, upper, size=(n_particles, DIM))
    vel = rng.uniform(-v_max, v_max, size=(n_particles, DIM))
    pbest_pos = pos.copy()
    pbest_fit = np.array([fitness(p) for p in pos])
    g_idx = pbest_fit.argmax()
    gbest_pos, gbest_fit = pbest_pos[g_idx].copy(), pbest_fit[g_idx]

    history = [gbest_fit]
    mean_history = [pbest_fit.mean()]
    worst_history = [pbest_fit.min()]
    print(f"Iter 00 | best CV-F1 = {gbest_fit:.4f}")

    for it in range(1, n_iter + 1):
        w = w_max - (w_max - w_min) * it / n_iter       # decreasing inertia weight
        for i in range(n_particles):
            r1, r2 = rng.rand(DIM), rng.rand(DIM)
            # Velocity update
            vel[i] = (w * vel[i]
                      + c1 * r1 * (pbest_pos[i] - pos[i])
                      + c2 * r2 * (gbest_pos   - pos[i]))
            vel[i] = np.clip(vel[i], -v_max, v_max)
            # Position update
            pos[i] = np.clip(pos[i] + vel[i], lower, upper)
            # Evaluate & update personal best
            f = fitness(pos[i])
            if f > pbest_fit[i]:
                pbest_fit[i], pbest_pos[i] = f, pos[i].copy()
                # Update global best
                if f > gbest_fit:
                    gbest_fit, gbest_pos = f, pos[i].copy()
        history.append(gbest_fit)
        mean_history.append(pbest_fit.mean())
        worst_history.append(pbest_fit.min())
        print(f"Iter {it:02d} | best CV-F1 = {gbest_fit:.4f} | mean pbest = {pbest_fit.mean():.4f}")
    return gbest_pos, gbest_fit, history, mean_history, worst_history

In [ ]:
t0 = time.time()
best_pos, best_fit, history, mean_history, worst_history = pso(n_particles=12, n_iter=20)
pso_time = time.time() - t0
best_params = decode(best_pos)
print(f"\nPSO finished in {pso_time:.1f}s")
print("Best hyperparameters:", best_params)
print(f"Best CV-F1: {best_fit:.4f}")

## 5. Train Optimised Model & Evaluate

In [ ]:
pso_rf = Pipeline(preprocess + [("rf", RandomForestClassifier(random_state=SEED, n_jobs=-1, **best_params))])
pso_rf.fit(X_train, y_train)
pso_metrics = evaluate(pso_rf, X_test, y_test)

imp = pd.DataFrame({"Baseline RF": base_metrics, "PSO-RF": pso_metrics})
imp["Δ (PSO − Base)"] = imp["PSO-RF"] - imp["Baseline RF"]
print("TEST-SET COMPARISON")
display(imp.round(4))

print("\nBest hyperparameters found by PSO vs defaults")
display(pd.DataFrame({"Default": [100, None, 2, 1, "sqrt"],
                      "PSO": [best_params[k] for k in param_names]}, index=param_names))

print("\nClassification report (PSO-RF):\n", classification_report(y_test, pso_rf.predict(X_test),
      target_names=["No disease", "Disease"]))

## 6. Visualisations

In [ ]:
# 6.1 PSO convergence curve (best / mean / worst particle vs. baseline)
plt.figure(figsize=(8, 4.8))
it = np.arange(len(history))
plt.plot(it, history,       "o-",  color="#1f77b4", lw=2, label="Best particle")
plt.plot(it, mean_history,  "s--", color="#ff7f0e", lw=2, label="Swarm mean")
plt.plot(it, worst_history, "^:",  color="grey",    lw=2, label="Worst particle")
plt.axhline(base_cv, color="red", ls="-.", lw=1.6, label=f"Baseline RF (CV F1 = {base_cv:.3f})")
plt.xticks(it[::2])
plt.xlabel("Iteration"); plt.ylabel("Fitness (5-fold CV F1-score)")
plt.title("PSO Convergence Curve (y-axis zoomed)"); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

In [ ]:
# 6.2 Baseline vs Optimised – metric comparison (test set)
GREY, GREEN = "#7f7f7f", "#2ca02c"
metrics = list(base_metrics.keys())
x = np.arange(len(metrics)); wd = 0.36
plt.figure(figsize=(10, 5))
b1 = plt.bar(x - wd/2, [base_metrics[m] for m in metrics], wd, color=GREY,  label="Baseline RF")
b2 = plt.bar(x + wd/2, [pso_metrics[m]  for m in metrics], wd, color=GREEN, label="PSO-optimised RF")
for bars in (b1, b2):
    for r in bars:
        plt.text(r.get_x() + r.get_width()/2, r.get_height() + 0.004, f"{r.get_height():.3f}",
                 ha="center", fontsize=8)
for i, m in enumerate(metrics):                       # change (PSO - baseline) above each pair
    d = pso_metrics[m] - base_metrics[m]
    plt.text(x[i], max(base_metrics[m], pso_metrics[m]) + 0.032, f"{d:+.3f}", ha="center",
             fontsize=10, fontweight="bold", color=("green" if d >= 0 else "red"))
plt.xticks(x, metrics); plt.ylim(0.70, 1.06); plt.ylabel("Test-set score (y-axis starts at 0.70)")
plt.title("Baseline vs PSO-Optimised Random Forest on held-out test set")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

In [ ]:
# 6.3 ROC curves: left = held-out test set, right = 5-fold CV predictions on all 303 patients
# Row 1 = full ROC, row 2 = zoom on the top-left corner where the two models differ
from sklearn.model_selection import cross_val_predict
curves = {}
for name, mdl in [("Baseline RF", baseline_rf), ("PSO-RF", pso_rf)]:
    p_test = mdl.predict_proba(X_test)[:, 1]
    p_cv   = cross_val_predict(mdl, X, y, cv=cv, method="predict_proba")[:, 1]
    curves[name] = [(y_test, p_test), (y, p_cv)]
colors = {"Baseline RF": GREY, "PSO-RF": GREEN}
titles = ["Held-out test set (61 patients)", "5-fold CV predictions (all 303 patients)"]

fig, ax = plt.subplots(2, 2, figsize=(13, 10))
for row in range(2):
    for col in range(2):
        a = ax[row, col]
        for name in curves:
            yt, p = curves[name][col]
            fpr, tpr, _ = roc_curve(yt, p)
            a.plot(fpr, tpr, color=colors[name], lw=2.2, label=f"{name} (AUC = {roc_auc_score(yt, p):.3f})")
        a.set_xlabel("False Positive Rate"); a.set_ylabel("True Positive Rate")
        if row == 0:
            a.plot([0, 1], [0, 1], "k--", alpha=0.5)
            a.set_title("ROC – " + titles[col])
        else:
            a.set_xlim(0, 0.4); a.set_ylim(0.6, 1.0)
            a.set_title("ROC (zoomed) – " + titles[col])
        a.legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# 6.4 Confusion matrices
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, (name, mdl) in zip(ax, [("Baseline RF", baseline_rf), ("PSO-RF", pso_rf)]):
    sns.heatmap(confusion_matrix(y_test, mdl.predict(X_test)), annot=True, fmt="d",
                cmap="Blues", cbar=False, ax=a,
                xticklabels=["No disease", "Disease"], yticklabels=["No disease", "Disease"])
    a.set_title(name); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout(); plt.show()

In [ ]:
# 6.5 Learning curves (train vs validation score vs training-set size)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for a, (name, mdl) in zip(ax, [("Baseline RF", baseline_rf), ("PSO-RF", pso_rf)]):
    sizes, tr, va = learning_curve(mdl, X_train, y_train, cv=cv, scoring="f1",
                                   train_sizes=np.linspace(0.2, 1.0, 6), n_jobs=-1, random_state=SEED)
    a.plot(sizes, tr.mean(1), "o-", label="Training F1")
    a.plot(sizes, va.mean(1), "s-", label="Validation F1")
    a.fill_between(sizes, va.mean(1) - va.std(1), va.mean(1) + va.std(1), alpha=0.15)
    a.set_title(f"Learning Curve – {name}"); a.set_xlabel("Training examples"); a.legend()
ax[0].set_ylabel("F1-score")
plt.tight_layout(); plt.show()

In [ ]:
# 6.6 Feature importance of the optimised model
imp_vals = pd.Series(pso_rf.named_steps["rf"].feature_importances_, index=X.columns).sort_values()
imp_vals.plot(kind="barh", figsize=(7, 5), color="teal")
plt.title("Feature Importance – PSO-RF"); plt.xlabel("Importance"); plt.tight_layout(); plt.show()

## 7. Robustness Check — Repeated Cross-Validation
A single 61-sample test set is small (one patient ≈ 1.6 % accuracy), so the two models are also compared with 5×5-fold repeated CV on the full dataset.

> **Caveat:** PSO tuned the hyperparameters using the *training* portion of the data, which is part of the full dataset used here. This table is therefore slightly optimistic for PSO-RF and should be read as a stability check, not an unbiased estimate.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
rows = []
for name, mdl in [("Baseline RF", baseline_rf), ("PSO-RF", pso_rf)]:
    for sc in ["accuracy", "f1", "roc_auc"]:
        s = cross_val_score(mdl, X, y, cv=rcv, scoring=sc, n_jobs=-1)
        rows.append({"Model": name, "Metric": sc, "Mean": s.mean(), "Std": s.std()})
display(pd.DataFrame(rows).pivot(index="Metric", columns="Model", values=["Mean", "Std"]).round(4))

## 7b. Second Optimiser — Genetic Algorithm (GA)
To check that the result does not depend on one optimiser, the same Random Forest search space and the same fitness function (5-fold CV F1 on the training set) are optimised with a **Genetic Algorithm written from scratch**.

**Operators:** tournament selection (k = 3), blend crossover (BLX-0.3, rate 0.8), Gaussian mutation (rate 0.2 per gene), elitism (top 2 survive). Population 12, 15 generations (about the same number of fitness evaluations as PSO).

Both optimisers use the same seed, so they start from the same random initial population — a fair like-for-like comparison.

In [ ]:
# ---------------- Genetic Algorithm ----------------
def ga(pop_size=12, n_gen=15, p_cx=0.8, p_mut=0.2, tour_k=3, n_elite=2, seed=SEED):
    rng = np.random.RandomState(seed)
    span = upper - lower
    pop = rng.uniform(lower, upper, size=(pop_size, DIM))
    fit = np.array([fitness(p) for p in pop])
    best_hist=[fit.max()]; mean_hist=[fit.mean()]; worst_hist=[fit.min()]
    print(f"Gen 00 | best CV-F1 = {fit.max():.4f}")
    for g in range(1, n_gen+1):
        order = np.argsort(-fit)
        elite_pos = pop[order[:n_elite]].copy(); elite_fit = fit[order[:n_elite]].copy()
        children = []
        while len(children) < pop_size - n_elite:
            # tournament selection
            def pick():
                idx = rng.choice(pop_size, tour_k, replace=False); return pop[idx[np.argmax(fit[idx])]]
            p1, p2 = pick(), pick()
            # blend crossover (BLX-0.3)
            if rng.rand() < p_cx:
                lo, hi = np.minimum(p1,p2), np.maximum(p1,p2); d = hi-lo
                c = rng.uniform(lo-0.3*d, hi+0.3*d)
            else:
                c = p1.copy()
            # gaussian mutation
            m = rng.rand(DIM) < p_mut
            c = c + m * rng.normal(0, 0.15*span)
            children.append(np.clip(c, lower, upper))
        children = np.array(children)
        cfit = np.array([fitness(c) for c in children])
        pop = np.vstack([elite_pos, children]); fit = np.concatenate([elite_fit, cfit])
        best_hist.append(fit.max()); mean_hist.append(fit.mean()); worst_hist.append(fit.min())
        print(f"Gen {g:02d} | best CV-F1 = {fit.max():.4f} | mean = {fit.mean():.4f}")
    b = fit.argmax()
    return pop[b], fit[b], best_hist, mean_hist, worst_hist


t0 = time.time()
ga_pos, ga_fit, ga_hist, ga_mean, ga_worst = ga(pop_size=12, n_gen=15)
ga_time = time.time() - t0
ga_params = decode(ga_pos)
print(f"\nGA finished in {ga_time:.1f}s")
print("Best hyperparameters:", ga_params)
print(f"Best CV-F1: {ga_fit:.4f}")

ga_rf = Pipeline(preprocess + [("rf", RandomForestClassifier(random_state=SEED, n_jobs=-1, **ga_params))])
ga_rf.fit(X_train, y_train)
ga_metrics = evaluate(ga_rf, X_test, y_test)

In [ ]:
# 7b.1 Three-way comparison: Baseline vs PSO-RF vs GA-RF
cmp3 = pd.DataFrame({"Baseline RF": base_metrics, "PSO-RF": pso_metrics, "GA-RF": ga_metrics})
print("TEST-SET COMPARISON (3 models)")
display(cmp3.round(4))
print("\nTraining 5-fold CV F1 (fitness):  baseline %.4f | PSO %.4f | GA %.4f" % (base_cv, best_fit, ga_fit))
print("Best hyperparameters  PSO:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in best_params.items()})
print("                      GA :", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in ga_params.items()})

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
# (a) metric comparison
GOLD = "#ff7f0e"
x = np.arange(len(metrics)); w = 0.26
for off, (nm, md, col) in zip([-w, 0, w], [("Baseline RF", base_metrics, GREY), ("PSO-RF", pso_metrics, GREEN), ("GA-RF", ga_metrics, GOLD)]):
    bars = ax[0].bar(x + off, [md[m] for m in metrics], w, color=col, label=nm)
    for r in bars:
        ax[0].text(r.get_x() + r.get_width()/2, r.get_height() + 0.004, f"{r.get_height():.3f}", ha="center", fontsize=7)
ax[0].set_xticks(x); ax[0].set_xticklabels(metrics); ax[0].set_ylim(0.70, 1.03)
ax[0].set_ylabel("Test-set score (y-axis starts at 0.70)"); ax[0].set_title("Baseline vs PSO-RF vs GA-RF (test set)"); ax[0].legend(loc="lower right")
# (b) convergence: PSO vs GA
ax[1].plot(history, "o-", color=GREEN, lw=2, label="PSO best (per iteration)")
ax[1].plot(mean_history, "s--", color=GREEN, alpha=0.5, label="PSO swarm mean")
ax[1].plot(ga_hist, "o-", color=GOLD, lw=2, label="GA best (per generation)")
ax[1].plot(ga_mean, "s--", color=GOLD, alpha=0.5, label="GA population mean")
ax[1].axhline(base_cv, color="red", ls="-.", lw=1.6, label=f"Baseline RF (CV F1 = {base_cv:.3f})")
ax[1].set_xlabel("Iteration / Generation"); ax[1].set_ylabel("Fitness (5-fold CV F1)")
ax[1].set_title("Convergence: PSO vs GA"); ax[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Summary of Results
The cell below prints the key numbers from this run so the conclusion can be written from them.

In [ ]:
# ---- Auto-generated summary of this run (numbers update if you re-run) ----
first_best = int(np.argmax(history))          # iteration where final best fitness was first reached
rep = pd.DataFrame(rows).pivot(index="Metric", columns="Model", values="Mean")
print("="*60)
print("SUMMARY OF THIS RUN")
print("="*60)
print(f"PSO search          : 12 particles x 20 iterations, {DIM} hyperparameters")
print(f"Best params         : {best_params}")
print(f"Training 5-fold CV F1: baseline {base_cv:.4f} -> PSO {best_fit:.4f}  (+{best_fit-base_cv:.4f})")
print(f"Best fitness first reached at iteration {first_best} of 20 (swarm converged / plateaued afterwards)")
print("\nTest-set change (PSO - Baseline):")
for m in base_metrics:
    d = pso_metrics[m] - base_metrics[m]
    print(f"  {m:10s}: {base_metrics[m]:.4f} -> {pso_metrics[m]:.4f}  ({d:+.4f})")
print("\nRepeated 5x5 CV on full data (mean):")
print(rep.round(4).to_string())
print(f"\nRuntime: baseline fit {baseline_time:.2f}s | PSO search {pso_time:.1f}s")

## 9. Conclusion
- A Random Forest was tuned with a from-scratch PSO (velocity/position updates, linearly decreasing inertia weight, personal and global bests) over 5 hyperparameters, using 5-fold CV F1 on the training set as fitness.
- **Training CV F1 improved clearly** over the default Random Forest (see summary above). On the held-out test set, accuracy, precision and F1 improved slightly, while recall and ROC-AUC were essentially unchanged or slightly lower.
- **The improvement is modest.** The test set has only 61 patients, and the repeated-CV comparison shows the two models perform almost identically on average. The gain is therefore best described as a better-regularised model (shallower trees, more of them, fewer features per split), not a large accuracy jump.
- **A second optimiser (GA) was run as a cross-check.** It reached essentially the same training CV F1 and very similar hyperparameters as PSO, so the gain over the default Random Forest is reproducible across two different nature-inspired algorithms, but neither produces a large jump on this small dataset.
- PSO's best fitness plateaued after roughly the first dozen iterations, which suggests premature convergence with a small swarm; more particles or iterations could be tried.
- **Limitations:** small dataset (303 samples), single train/test split, PSO stochasticity, and the repeated-CV table is slightly optimistic for PSO-RF because tuning used part of the same data.
- **Future work:** other optimisers (GA, DE, Firefly), more datasets (Hungarian, VA, Switzerland), nested cross-validation for an unbiased estimate, and PSO-based feature selection.